In [8]:
import os
# Run from project folder so relative paths like 'ddc-master_patient_catalog.parquet' work
os.chdir(r'C:\AI\chi-data-dictionary-catalog')

In [9]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///:memory:


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [11]:
import duckdb

# Query with absolute path so the table always displays (%%sql often doesn't show results)
path = os.path.join(os.getcwd(), 'ddc-master_patient_catalog.parquet')
df = duckdb.query(f"SELECT * FROM read_parquet('{path}') LIMIT 10").df()
df

,Semantic ID,USCDI Element,USCDI Description,Classification,Ruleset Category,Privacy/Security
0,Patient.name_first,First Name,Patient's first/given name,Master Demographics,Static Identity,
1,Patient.name_last,Last Name,Patient's last/family name,Master Demographics,Static Identity,
2,Patient.name_middle,Middle Name,Patient's middle name(s),Master Demographics,Static Identity,
3,Patient.name_suffix,Suffix,"Name suffix (Jr., Sr., III, etc.)",Master Demographics,Static Identity,
4,Patient.name_prev,Previous Name,Prior legal names,Master Demographics,Static Identity,
5,Patient.birth_date,Date of Birth,Patient's birth date,Master Demographics,Static Identity,
6,Patient.death_date,Date of Death,Patient's date of death,Master Demographics,Static Identity (Vital Status),
7,Patient.birth_sex,Sex,Sex assigned at birth,Master Demographics,Static Identity,
8,Patient.gender_id,Gender Identity,Patient's current gender identity,Master Demographics,Dynamic Identity,
9,Patient.sexual_orient,Sexual Orientation,Patient's sexual orientation,Master Demographics,Dynamic Identity,Sensitive/SOGI: Restricted access; usually req...


In [12]:
# Multiline SQL: join catalog and dictionary on semantic_id (snake_case columns)
path_cat = os.path.join(os.getcwd(), 'ddc-master_patient_catalog.parquet')
path_dict = os.path.join(os.getcwd(), 'ddc-master_patient_dictionary.parquet')

df_joined = duckdb.query(f"""
    SELECT c.*, d.chi_survivorship_logic, d.fhir_r4_path, d.fhir_data_type
    FROM read_parquet('{path_cat}') c
    JOIN read_parquet('{path_dict}') d
      ON c.semantic_id = d.semantic_id
    LIMIT 10
""").df()
df_joined

,Semantic ID,USCDI Element,USCDI Description,Classification,Ruleset Category,Privacy/Security,SHIE Survivorship Logic,FHIR R4 Path,FHIR Data Type
0,Patient.name_first,First Name,Patient's first/given name,Master Demographics,Static Identity,,"Most complete (full name > initial), most rece...",Patient.name.given,HumanName.given[0] (string)
1,Patient.name_last,Last Name,Patient's last/family name,Master Demographics,Static Identity,,"Most complete, most recent user entry as tiebr...",Patient.name.family,HumanName.family (string)
2,Patient.name_middle,Middle Name,Patient's middle name(s),Master Demographics,Static Identity,,"Complete middle name > initial > null, most re...",Patient.name.given,HumanName.given[1+] (string array)
3,Patient.name_suffix,Suffix,"Name suffix (Jr., Sr., III, etc.)",Master Demographics,Static Identity,,"Most recent valid entry, ignore if inconsisten...",Patient.name.suffix,HumanName.suffix (0..*)
4,Patient.name_prev,Previous Name,Prior legal names,Master Demographics,Static Identity,,Maintain all historical names with effective d...,Patient.name (with period.start/end),HumanName with Period (0..*)
5,Patient.birth_date,Date of Birth,Patient's birth date,Master Demographics,Static Identity,,Most reliable source (birth certificate > gove...,Patient.birthDate,date
6,Patient.death_date,Date of Death,Patient's date of death,Master Demographics,Static Identity (Vital Status),,Most reliable source (death certificate > vita...,Patient.deceased[x],Choice: deceasedBoolean OR deceasedDateTime
7,Patient.birth_sex,Sex,Sex assigned at birth,Master Demographics,Static Identity,,"Most reliable source, medical record > governm...",Patient.extension(us-core-birthsex),Extension ? code (M/F/UNK)
8,Patient.gender_id,Gender Identity,Patient's current gender identity,Master Demographics,Dynamic Identity,,"Most recent self-reported value, timestamp req...",Observation.valueCodeableConcept (LOINC 76691-5),Observation resource (US Core Observation Gend...
9,Patient.sexual_orient,Sexual Orientation,Patient's sexual orientation,Master Demographics,Dynamic Identity,Sensitive/SOGI: Restricted access; usually req...,"Most recent self-reported value, timestamp req...",Observation.valueCodeableConcept (LOINC 76690-7),Observation resource (US Core Observation Sexu...


In [1]:
# Optional: create a minimal ddc-ccda_catalog.parquet (2 rows) for quick testing.
# For the full CCD catalog (34+ rows), run instead:
#   python scripts/build_ccda_catalog_from_mapping.py
# (reads data/ccd_to_semantic_id_mapping.csv)

import pandas as pd
from pathlib import Path

root = Path('.')

ccda_rows = [
    {
        'message_format': 'CCD',
        'section_name': 'Demographics',
        'entry_type': 'Patient Name',
        'xml_path': '/ClinicalDocument/recordTarget/patientRole/patient/name',
        'semantic_id': 'Patient.name_last',
        'fhir_r4_path': 'Patient.name.family',
        'notes': 'CCD demographics name element (family).',
    },
    {
        'message_format': 'CCD',
        'section_name': 'Demographics',
        'entry_type': 'Date of Birth',
        'xml_path': '/ClinicalDocument/recordTarget/patientRole/patient/birthTime',
        'semantic_id': 'Patient.birth_date',
        'fhir_r4_path': 'Patient.birthDate',
        'notes': 'CCD birthTime element.',
    },
]

ccda_df = pd.DataFrame(ccda_rows)
(root / 'ddc-ccda_catalog.parquet').unlink(missing_ok=True)
ccda_df.to_parquet(root / 'ddc-ccda_catalog.parquet', index=False)
print('Wrote ddc-ccda_catalog.parquet')

Wrote ddc-hl7_adt_catalog.parquet and ddc-ccda_catalog.parquet


In [2]:
#  Export master catalog and dictionary Parquet files to pipe-delimited CSVs.

import duckdb

con = duckdb.connect()

con.execute("""
    COPY (SELECT * FROM read_parquet('ddc-master_patient_catalog.parquet'))
    TO 'ddc-master_patient_catalog_pipe.csv'
    (FORMAT CSV, DELIMITER '|', HEADER TRUE);
""")

con.execute("""
    COPY (SELECT * FROM read_parquet('ddc-master_patient_dictionary.parquet'))
    TO 'ddc-master_patient_dictionary_pipe.csv'
    (FORMAT CSV, DELIMITER '|', HEADER TRUE);
""")

print("Wrote ddc-master_patient_catalog_pipe.csv and ddc-master_patient_dictionary_pipe.csv")

Wrote ddc-master_patient_catalog_pipe.csv and ddc-master_patient_dictionary_pipe.csv
